# Get Pagination Links
This notebook contains functions that:
1) Parse the html document of first page of works for any given Archive Of Our Own fandom (e.g.: https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?page=1)
2) Return a list of pagination links from the pagination menu at the top of the first page of works

# Import libraries

In [1]:
from bs4 import BeautifulSoup
import requests
import re

# Parse "first page of works" html doc

In [2]:
def get_html_doc(url):
    """Takes in the url for the first page of works for any given Ao3 fandom and returns a BeautifulSoup object from that page."""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

# Get pagination links list

In [8]:
def get_all_urls(soup):
    """Takes in BeautifulSoup object from get_html_doc and returns a list of all links in the object."""
    links = soup.find_all('a')
    for link in links:
        link = link.get('href')
    return links

In [41]:
def get_urls_only(links_list):
    """Takes in a list of all links from get_all_urls and returns a list of only the URL strings."""
    # use regex to get just URLs
    urls_only = []
    pattern = r'\"(.*?)\"'
    for i in range(len(links_list)):
        link_str = str(links_list[i])
        match = re.search(pattern, link_str)
        if match:
            urls_only.append(match.group(1))
        else:
            continue

    # update URLs if they do not include 'http://archiveofourown.org' at the start
    for i in range(len(urls_only)):
        if urls_only[i][0] == '/':
            urls_only[i] = 'https://archiveofourown.org' + urls_only[i]
        else:
            continue

    return urls_only

In [9]:
# TODO - delete once testing
def get_urls_only_OLD(links_list):
    """Takes in a list of all links from get_all_urls and returns a list of only the URL strings."""
    # use regex to get just URLs
    urls_only = []
    pattern = r'\"(.*?)\"'
    for i in range(len(links_list)):
        link_str = str(links_list[i])
        match = re.search(pattern, link_str)
        if match:
            urls_only.append(match.group(1))
        else:
            continue
    return urls_only

In [15]:
# TODO - delete once testing
def update_urls_only(urls_only):
    """Updates URLS from get_urls_only if they do not include 'http://archiveofourown.org' at the start"""
    for i in range(len(urls_only)):
        if urls_only[i][0] == '/':
            urls_only[i] = 'https://archiveofourown.org' + urls_only[i]
        else:
            continue
    return urls_only

In [16]:
def get_all_ol(soup):
    """Takes in BeautifulSoup object from get_html_doc and returns a list of all elements of the class 'ol'. This is a step in finding all pagination links for a given Ao3 fandom's works."""
    all_ol = soup.find_all('ol')
    all_ol = list(all_ol)
    return all_ol

In [31]:
def get_pagination_links(all_ol):
    """Takes in all 'ol' elements from get_all_ol and returns a list of raw pagination links for that fandom's works."""
    pagination1 = all_ol[0]

    # get only href items from all_ol
    pagination_links = pagination1.find_all('a')
    pagination_links_list = []
    for link in pagination_links:
        print(link.get('href'))
        pagination_links_list.append(link.get('href'))

    return pagination_links_list

In [32]:
def update_pagination_links_list(list):
    """Takes in a list of raw pagination links from get_pagination_links and returns a more parseable list of links to scrape."""
    # get second to last item in list
    pagination_idx = len(list) - 2
    second_to_last_link = list[pagination_idx]

    # get number associated with second-to-last link
    last_page = second_to_last_link.rsplit('=',1)
    last_page_num = last_page[1]

    # generate list of pagination links to scrape
    links_to_scrape = []
    last_page_num1 = int(last_page_num)
    link_str = last_page[0] + '='
    for i in range(1,last_page_num1+1):
        num = str(i)
        full_link = 'https://archiveofourown.org' + link_str + num
        links_to_scrape.append(full_link)

    return links_to_scrape

In [42]:
def save_links_to_txt(file_path, links_to_scrape):
    """Writes the list of pagination links from update_pagination_links_list to a .txt file."""

    # Using "with open" syntax to automatically close the file
    with open(file_path, 'w') as file:
        # Join the list elements into a single string with a newline character
        data_to_write = '\n'.join(links_to_scrape)

        # Write the data to the file
        file.write(data_to_write)

    print(f"The list of links to scrape has been written to {file_path}.")

# Testing

In [20]:
# test get_html_doc
candelaobscuraurl = 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?page=1'
soup = get_html_doc(candelaobscuraurl)

In [36]:
all_urls = get_all_urls(soup)
all_urls_updated = get_urls_only(all_urls)
all_ol = get_all_ol(soup)
pagination_links = get_pagination_links(all_ol)
links_to_scrape = update_pagination_links_list(pagination_links)
print("There are: ", str(len(links_to_scrape)), " pagination links to scrape for the Candela Obscura fandom.")
print("Pagination links:")

None
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=2
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=4
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=5
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=6
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=7
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=11
/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=2
There are:  11  pagination links to scrape for the Candela Obscura fandom.
Pagination links:
['https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%2

In [37]:
print(links_to_scrape)

['https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=1', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=2', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=3', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=4', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=5', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=6', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=7', 'https://archiveofourown.org/tags/Candela%20Obscura%20(Critical%20Role%20Web%20Series)/works?view_adult=true&page=8', 'https://archiveofourown.org/tags/Candela%20Obscura%20(

In [43]:
# save pagination links to text file
file_path = "../data/txt_files/pagination_links.txt"
save_links_to_txt(file_path, links_to_scrape)

The list of links to scrape has been written to ../data/txt_files/pagination_links.txt.


Now that I have a list of pages for works associated with the Candela Obscura fandom, I can take that list and scrape each page for links to fanworks. See Step2_get_fanwork_links.ipynb.

In [19]:
def get_fanwork_links(filepath):
    """Takes in a txt file of pagination links and returns a list."""
    fanwork_links = open(filepath).readlines()
    for i in range(len(fanwork_links)):
        fanwork_links[i] = fanwork_links[i].replace("\n","")
    return fanwork_links

In [21]:
def remove_chapter_links(work_links):
    """Takes in a list of pared down fanwork URLs from pare_down_work_links and removes any links to individual fanwork chapters."""
    works_only_final = work_links
    for i in range(len(work_links)):
        current_link = work_links[i]
        if 'chapters' in current_link:
            works_only_final.remove(current_link)
        else:
            continue
    return works_only_final

In [23]:
## get pagination links to scrape
pagination_links1 = get_fanwork_links('/data/txt_files/fanwork_links1.txt')
pagination_links2 = get_fanwork_links('/data/txt_files/fanwork_links2.txt')
pagination_links3 = get_fanwork_links('/data/txt_files/fanwork_links3.txt')
pagination_links4 = get_fanwork_links('/data/txt_files/fanwork_links4.txt')
pagination_links5 = get_fanwork_links('/data/txt_files/fanwork_links5.txt')
pagination_links6 = get_fanwork_links('/data/txt_files/fanwork_links6.txt')
pagination_links7 = get_fanwork_links('/data/txt_files/fanwork_links7.txt')
pagination_links8 = get_fanwork_links('/data/txt_files/fanwork_links8.txt')
pagination_links9 = get_fanwork_links('/data/txt_files/fanwork_links9.txt')
pagination_links10 = get_fanwork_links('/data/txt_files/fanwork_links10.txt')
pagination_links11 = get_fanwork_links('/data/txt_files/fanwork_links11.txt')

In [27]:
pagination_links1 = remove_chapter_links(pagination_links1)

In [28]:
print(pagination_links1)

['https://archiveofourown.org/works/80602681', 'https://archiveofourown.org/works/51518734', 'https://archiveofourown.org/works/51010177', 'https://archiveofourown.org/works/80156821', 'https://archiveofourown.org/works/74479381', 'https://archiveofourown.org/works/56574286', 'https://archiveofourown.org/works/75864761', 'https://archiveofourown.org/works/73925776', 'https://archiveofourown.org/works/69525656', 'https://archiveofourown.org/works/69696606', 'https://archiveofourown.org/works/69266581', 'https://archiveofourown.org/works/68678736', 'https://archiveofourown.org/works/59798779', 'https://archiveofourown.org/works/55118308', 'https://archiveofourown.org/works/66001534', 'https://archiveofourown.org/works/62477026', 'https://archiveofourown.org/works/62036794', 'https://archiveofourown.org/works/60847894', 'https://archiveofourown.org/works/51300544', 'https://archiveofourown.org/works/56817901']


In [30]:
pagination_links2 = remove_chapter_links(pagination_links2)

In [32]:
pagination_links3 = remove_chapter_links(pagination_links3)

In [ ]:
# remove chapters links from each pagination link
for i in range(len(pagination_links7)):
    current_link = pagination_links7[i]
    if 'chapters' in current_link:
        pagination_links7.remove(current_link)
    else:
        continue